In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 116000,
    "squadv2": 130000,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

In [3]:
prompt_type = "direct_qa"

In [4]:
def extract_alpha(str):
    # use regex to extract the number after "alpha="
    import re
    match = re.search(r"alpha=([\d.]+)", str)
    if match:        return float(match.group(1))
    else:        return None    

def extract_beta(str):
    # use regex to extract the number after "beta="
    import re
    match = re.search(r"beta=([\d.]+)", str)
    if match:        return float(match.group(1))
    else:        return None   

In [5]:
results_dir = f"/hdd/ivny/{prompt_type}_in_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if os.path.exists(os.path.join(leaf_dir, "calibration_details.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_details.csv"))

        original_alpha = csv_data["original_lc"].apply(extract_alpha)
        original_beta = csv_data["original_lc"].apply(extract_beta)
        lc_calibrated_alpha = csv_data["calibrated_lc_rewritten_lc"].apply(extract_alpha)
        lc_callable_alpha = csv_data["calibrated_lc_rewritten_lc"].apply(extract_beta)
        tp_calibrated_alpha = csv_data["calibrated_tp_rewritten_lc"].apply(extract_alpha)
        tp_callable_alpha = csv_data["calibrated_tp_rewritten_lc"].apply(extract_beta)
        su_calibrated_alpha = csv_data["calibrated_su_rewritten_lc"].apply(extract_alpha)
        su_callable_alpha = csv_data["calibrated_su_rewritten_lc"].apply(extract_beta)
        accuracy = csv_data["accuracy"].tolist()

        record = {
            "model_name": model_name_map.get(model_name, model_name),
            "dataset_name": dataset_map.get(dataset_name, dataset_name),
            "original_alpha": original_alpha,
            "original_beta": original_beta,
            "lc_calibrated_alpha": lc_calibrated_alpha,
            "lc_callable_alpha": lc_callable_alpha,
            "tp_calibrated_alpha": tp_calibrated_alpha,
            "tp_callable_alpha": tp_callable_alpha,
            "su_calibrated_alpha": su_calibrated_alpha,
            "su_callable_alpha": su_callable_alpha,
            "accuracy": accuracy
        }

        all_records.append(record)

    else:
        print(model_name, dataset_name, "missing calibration_details.pkl")

In [6]:
import os
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgba
from scipy.special import betaln, psi

def _clean_accuracy(values):
    cleaned = []
    for val in values:
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        if val == "":
            val = 0.0
        try:
            cleaned.append(float(val))
        except (ValueError, TypeError):
            continue
    return np.asarray(cleaned, dtype=float)

def _align_and_clean(alpha_vals, beta_vals, accuracy_vals):
    a = np.asarray(alpha_vals, dtype=float)
    b = np.asarray(beta_vals, dtype=float)
    y = np.asarray(accuracy_vals, dtype=float)

    n = min(len(a), len(b), len(y))
    a = a[:n]
    b = b[:n]
    y = y[:n]

    mask = np.isfinite(a) & np.isfinite(b) & np.isfinite(y) & (a > 0) & (b > 0)
    return a[mask], b[mask], y[mask]

def _generalised_ece_curve(
    alpha_vals,
    beta_vals,
    accuracy_vals,
    num_bins=10,
    num_samples=1500,
    n_bootstrap=800,
    ci_percentiles=(5, 95),
    seed=None,
    ):
    a, b, y = _align_and_clean(alpha_vals, beta_vals, accuracy_vals)
    n_obs = len(y)
    if n_obs == 0:
        return {
            "gece": np.nan,
            "x": np.array([]),
            "acc": np.array([]),
            "lower": np.array([]),
            "upper": np.array([]),
            "density": np.array([]),
            "n": 0,
        }

    rng = np.random.default_rng(seed)
    all_samples = rng.beta(a[:, None], b[:, None], size=(n_obs, num_samples))

    bins = np.linspace(0.0, 1.0, num_bins + 1)
    bin_indices = np.digitize(all_samples, bins) - 1
    bin_indices = np.clip(bin_indices, 0, num_bins - 1)

    rm = np.zeros(num_bins)
    pm = np.zeros(num_bins)
    gm = np.zeros(num_bins)

    plot_x = []
    plot_acc = []
    plot_lower = []
    plot_upper = []
    plot_density = []

    for m in range(num_bins):
        mask = (bin_indices == m)
        p_nm = np.mean(mask, axis=1)
        pm[m] = np.sum(p_nm)

        if pm[m] <= 0:
            continue

        rm[m] = np.sum(p_nm * y) / pm[m]
        bin_samples = all_samples[mask]
        gm[m] = np.mean(bin_samples) if bin_samples.size > 0 else 0.0

        ws = p_nm / pm[m]
        ws_sum = ws.sum()
        if ws_sum > 0:
            ws = ws / ws_sum

        bootstrap_estimates = []
        if n_obs > 0 and n_bootstrap > 0:
            for _ in range(n_bootstrap):
                idx = rng.integers(0, n_obs, size=n_obs)
                ws_b = ws[idx]
                ys_b = y[idx]
                ws_b_sum = ws_b.sum()
                if ws_b_sum > 0:
                    ws_b = ws_b / ws_b_sum
                    bootstrap_estimates.append(np.sum(ws_b * ys_b))

        if len(bootstrap_estimates) > 0:
            lower, upper = np.percentile(bootstrap_estimates, ci_percentiles)
        else:
            lower = rm[m]
            upper = rm[m]

        plot_x.append(gm[m])
        plot_acc.append(rm[m])
        plot_lower.append(lower)
        plot_upper.append(upper)
        plot_density.append(pm[m] / n_obs)

    gece = np.sum((pm / n_obs) * np.abs(rm - gm))

    return {
        "gece": float(gece),
        "x": np.asarray(plot_x),
        "acc": np.asarray(plot_acc),
        "lower": np.asarray(plot_lower),
        "upper": np.asarray(plot_upper),
        "density": np.asarray(plot_density),
        "n": int(n_obs),
    }

def _faithfulness_divergence(alpha_vals, beta_vals, accuracy_vals):
    a, b, y = _align_and_clean(alpha_vals, beta_vals, accuracy_vals)
    if len(y) == 0:
        return np.nan

    a_post = a + y
    b_post = b + (1.0 - y)

    kl_vals = (
        betaln(a, b) - betaln(a_post, b_post)
        + (a_post - a) * psi(a_post)
        + (b_post - b) * psi(b_post)
        - (a_post + b_post - a - b) * psi(a_post + b_post)
    )

    fd_vals = np.maximum(0.0, (a + b + 1e-8) * kl_vals)
    fd_vals = fd_vals[np.isfinite(fd_vals)]
    if len(fd_vals) == 0:
        return np.nan
    return float(np.mean(fd_vals))

def _add_density_shaded_band(ax, x, lower, upper, density, color="#2ca02c"):
    if len(x) < 2:
        return

    order = np.argsort(x)
    x_sorted = x[order]
    lower_sorted = lower[order]
    upper_sorted = upper[order]
    density_sorted = density[order]

    dense_x = np.linspace(0.0, 1.0, 220)
    dense_lower = np.interp(dense_x, x_sorted, lower_sorted, left=lower_sorted[0], right=lower_sorted[-1])
    dense_upper = np.interp(dense_x, x_sorted, upper_sorted, left=upper_sorted[0], right=upper_sorted[-1])
    dense_density = np.interp(dense_x, x_sorted, density_sorted, left=density_sorted[0], right=density_sorted[-1])

    if np.max(dense_density) > 0:
        dense_density = dense_density / np.max(dense_density)

    verts = []
    colors = []
    for i in range(len(dense_x) - 1):
        local_density = 0.5 * (dense_density[i] + dense_density[i + 1])
        alpha = 0.08 + 0.65 * float(np.clip(local_density, 0.0, 1.0))
        verts.append([
            (dense_x[i], dense_lower[i]),
            (dense_x[i], dense_upper[i]),
            (dense_x[i + 1], dense_upper[i + 1]),
            (dense_x[i + 1], dense_lower[i + 1]),
        ])
        colors.append(to_rgba(color, alpha=alpha))

    band = PolyCollection(verts, facecolors=colors, edgecolors="none")
    ax.add_collection(band)

def _plot_reliability_diagram(ax, alpha_vals, beta_vals, accuracy_vals, panel_title):
    curve = _generalised_ece_curve(
        alpha_vals,
        beta_vals,
        accuracy_vals,
        ci_percentiles=(2.5, 97.5),
    )
    fd = _faithfulness_divergence(alpha_vals, beta_vals, accuracy_vals)

    ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.7, linewidth=1)
    _add_density_shaded_band(ax, curve["x"], curve["lower"], curve["upper"], curve["density"])
    if len(curve["x"]) > 0:
        order = np.argsort(curve["x"])
        ax.plot(curve["x"][order], curve["acc"][order], "o-", color="black", markersize=1.5, linewidth=1)

    ax.text(
        0.97,
        0.03,
        f"ECE={curve['gece']:.3f}\nFD={fd:.3f}",
        transform=ax.transAxes,
        va="bottom",
        ha="right",
        fontsize=7,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 2.0},
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_title(panel_title, fontsize=8)

if len(all_records) == 0:
    raise ValueError("No records found in all_records. Run the data-loading cell first.")

num_cols = 4
dataset_groups = {}
for rec in all_records:
    ds = rec["dataset_name"]
    dataset_groups.setdefault(ds, []).append(rec)

saved_paths = []
for dataset_name in sorted(dataset_groups.keys()):
    dataset_records = dataset_groups[dataset_name]
    num_rows = len(dataset_records)

    fig, axes = plt.subplots(
        num_rows,
        num_cols,
        figsize=(1.75 * num_cols, max(1.75 * num_rows, 3.0)),
        squeeze=False,
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for r, record in enumerate(dataset_records):
        y = _clean_accuracy(record["accuracy"])

        panel_data = [
            (record["original_alpha"], record["original_beta"], "Original"),
            (record["lc_calibrated_alpha"], record["lc_callable_alpha"], "Calibrated LC as Signal"),
            (record["tp_calibrated_alpha"], record["tp_callable_alpha"], "Calibrated TP as Signal"),
            (record["su_calibrated_alpha"], record["su_callable_alpha"], "Calibrated SU as Signal"),
        ]

        for c, (a_vals, b_vals, panel_name) in enumerate(panel_data):
            ax = axes[r, c]
            _plot_reliability_diagram(ax, a_vals, b_vals, y, panel_name)

            if c == 0:
                ax.set_ylabel(f"{record['model_name']}\nAccuracy", fontsize=8)

            ax.set_xlabel("Mean Confidence", fontsize=8)

    # fig.suptitle(f"{dataset_name}: In-domain reliability diagrams", fontsize=11)
    safe_name = str(dataset_name).lower().replace(" ", "_").replace("/", "-")
    out_path = f"in_domain_reliability_diagrams_{safe_name}.pdf"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    saved_paths.append(out_path)
    plt.close(fig)

print("Saved files:")
for path in saved_paths:
    print(path)

Saved files:
in_domain_reliability_diagrams_mmlu.pdf
in_domain_reliability_diagrams_squad2.0.pdf
in_domain_reliability_diagrams_truthfulqa.pdf
